In [2]:
import pandas as pd
vitals_df = pd.read_csv("../data/processed/vitals_final.csv")

ed_only = vitals_df[vitals_df['charttime_ed'].isna() == False]
ed_only = ed_only.drop(columns = ['charttime_lab', 'Lab Oxygen Saturation %', 'Lab Temperature (C)', 'Hosp BMI (kg/m2)', 'Hosp Height (Inches)', 'Hosp Weight (Lbs)', 'Hosp Blood Pressure'])

ecg_df = pd.read_csv('../data/processed/ecg_data.csv')
ecg_ed_only = ecg_df[ecg_df['in_ed'] == 1]

clinical_encounters = pd.read_csv('../data/processed/clinical_encounters_extracted.csv')
ed_only_encounters = clinical_encounters[clinical_encounters['ed_stay_id'].isna() == False]

ed_only_encounters = ed_only_encounters.drop(columns = ['icu_outtime'])
ecg_ed_only = ecg_ed_only.drop(columns = ['icu_within_12hrs', 'icu_within_24hrs'])
ed_only_encounters = ed_only_encounters.drop(columns = ['icu_stay_id', 'icu_stay_id', 'icu_intime', 'icu_count'])

join1 = ecg_ed_only.merge(ed_only_encounters, how = 'left', on = ['subject_id', 'ed_stay_id'])
join1 = ecg_ed_only.merge(ed_only_encounters, how = 'left', on = ['subject_id', 'ed_stay_id'])

ecg = ecg_ed_only.copy()
enc = ed_only_encounters.copy()
vitals = ed_only.copy()

ecg['ecg_time'] = pd.to_datetime(ecg['ecg_time'])
vitals['charttime_ed'] = pd.to_datetime(vitals['charttime_ed'])
enc['ed_intime'] = pd.to_datetime(enc['ed_intime'])
enc['ed_outtime'] = pd.to_datetime(enc['ed_outtime'])

ecg = ecg[
    (ecg['in_ed'] == 1) &
    (ecg['ed_stay_id'].notna()) &
    (ecg['ecg_time'].notna())
].copy()

ecg = ecg.merge(
    enc[['subject_id', 'ed_stay_id', 'anchor_age', 'gender', 'race', 'ed_intime', 'ed_outtime', 'label_active_infection_nonsepsis',
       'label_acute_heart_failure',
       'label_acute_hypercapnic_respiratory_failure',
       'label_acute_hypoxic_respiratory_failure', 'label_acute_kidney_injury',
       'label_acute_on_chronic_heart_failure', 'label_anemia',
       'label_aortic_disease', 'label_ards', 'label_asthma_exacerbation',
       'label_atrial_fibrillation_flutter', 'label_atrioventricular_block',
       'label_bradyarrhythmias_heart_block', 'label_bundle_branch_block',
       'label_cardiac_arrest', 'label_cardiogenic_shock', 'label_cardiomegaly',
       'label_cardiomyopathy', 'label_chronic_heart_failure',
       'label_chronic_ischemic_heart_disease', 'label_chronic_kidney_disease',
       'label_chronic_total_occlusion', 'label_copd_exacerbation',
       'label_coronary_aneurysm_dissection', 'label_coronary_artery_disease',
       'label_deep_vein_thrombosis', 'label_diabetes_mellitus',
       'label_dialysis_dependence', 'label_electrolyte_disturbance',
       'label_end_stage_heart_failure', 'label_endocarditis',
       'label_gi_bleed_or_hepatic_failure', 'label_hemorrhagic_stroke',
       'label_high_output_heart_failure', 'label_hypertension',
       'label_hypertensive_emergency', 'label_ischemic_stroke',
       'label_long_qt_syndrome', 'label_myocarditis',
       'label_noncardiogenic_pulmonary_edema', 'label_nstemi',
       'label_other_arrhythmias', 'label_other_conduction_disorders',
       'label_other_heart_disease', 'label_pericarditis_pericardial_disease',
       'label_peripheral_vascular_disease', 'label_pleural_effusion',
       'label_pneumonia', 'label_pneumothorax',
       'label_premature_depolarization', 'label_pulmonary_embolism',
       'label_pulmonary_hypertension', 'label_rare_cardiac_structural',
       'label_right_heart_failure', 'label_sepsis', 'label_septic_shock',
       'label_sick_sinus_syndrome', 'label_stable_angina', 'label_stemi',
       'label_subsequent_mi', 'label_supraventricular_tachycardia',
       'label_tia', 'label_unspecified_cardiac', 'label_unstable_angina',
       'label_valvular_heart_disease', 'label_ventricular_arrhythmias',
       'is_cardiovascular']],
    on=['subject_id', 'ed_stay_id'],
    how='left'
)

vitals = vitals.rename(columns={'stay_id': 'ed_stay_id'})
vitals = vitals[
    [
        'subject_id',
        'ed_stay_id',
        'charttime_ed',
        'ED Temperature (F)',
        'ED Heart Rate',
        'ED Respitory Rate',
        'ED Oxygen Saturation %',
        'ED sbp',
        'ED dbp'
    ]
].copy()

ecg = ecg.sort_values('ecg_time')
vitals = vitals.sort_values('charttime_ed')

ecg_vitals = pd.merge_asof(
    ecg,
    vitals,
    left_on='ecg_time',
    right_on='charttime_ed',
    by=['subject_id', 'ed_stay_id'],
    direction='backward',
    tolerance=pd.Timedelta('4H')  # optional but recommended
)

ecg_vitals['minutes_since_vitals'] = (
    ecg_vitals['ecg_time'] - ecg_vitals['charttime_ed']
).dt.total_seconds() / 60

using_this = ecg_vitals[ecg_vitals['charttime_ed'].isna() == False]
using_this = using_this.drop(columns = ['icu_stay_id',
 'in_hosp',
 'in_ed',
 'in_icu',
 'cart_id', 'full_report'])
data = using_this.copy()


/tmp/ipykernel_570/1896096152.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  vitals_df = pd.read_csv("../data/processed/vitals_final.csv")
/tmp/ipykernel_570/1896096152.py:10: DtypeWarning: Columns (3,4,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical_encounters = pd.read_csv('../data/processed/clinical_encounters_extracted.csv')


In [3]:
data['rr_interval']

1        674
2        667
3        698
19       822
36       606
        ... 
75173    789
75176    577
75177    769
75188    545
75205    882
Name: rr_interval, Length: 8816, dtype: int64

In [4]:
ecg_numeric = [
    'rr_interval',
    'p_onset', 'p_end',
    'qrs_onset', 'qrs_end',
    't_end',
    'p_axis', 'qrs_axis', 't_axis'
]

using_this['qrs_duration'] = using_this['qrs_end'] - using_this['qrs_onset']
using_this['pr_interval'] = using_this['qrs_onset'] - using_this['p_onset']
using_this['qt_proxy'] = using_this['t_end'] - using_this['qrs_onset']

engineered_ecg = ['qrs_duration', 'pr_interval', 'qt_proxy']

vital_features = [
    'ED Heart Rate',
    'ED sbp',
    'ED dbp',
    'ED Respitory Rate',
    'ED Oxygen Saturation %',
    'ED Temperature (F)',
    'minutes_since_vitals'
]

demo_numeric = ['anchor_age']
demo_categorical = ['gender', 'race']



In [5]:
REPORT_LABELS = [col for col in data.columns if col.startswith('report_')]

# Drop labels with zero variance
Y_report = data[REPORT_LABELS].astype(int)

print("Original Y_report shape:", Y_report.shape)
print("Unique counts per label:\n", Y_report.nunique())


Y_report = Y_report.loc[:, Y_report.nunique() > 1]

# Features for report model
X_report = data[ecg_numeric + vital_features + demo_numeric + demo_categorical]

# -----------------------------
# 5. Prepare DIAGNOSIS_LABELS
# -----------------------------
DIAGNOSIS_LABELS = [col for col in data.columns if col.startswith('label_')]

# Remove extremely rare labels (less than 20 positives)
Y_diag = data[DIAGNOSIS_LABELS].astype(int)
label_counts = Y_diag.sum()
keep_diag_labels = label_counts[label_counts >= 20].index
Y_diag = Y_diag[keep_diag_labels]

# Features for diagnosis model: only ECG + vitals + demographics (no report labels)
X_diag = data[ecg_numeric + vital_features + demo_numeric + demo_categorical]

from sklearn.model_selection import GroupShuffleSplit
# -----------------------------
# 6. Train/test split
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_diag, Y_diag, groups=data['subject_id']))

def split(X, Y):
    return X.iloc[train_idx], X.iloc[test_idx], Y.iloc[train_idx], Y.iloc[test_idx]

Xr_train, Xr_test, Yr_train, Yr_test = split(X_report, Y_report)
Xd_train, Xd_test, Yd_train, Yd_test = split(X_diag, Y_diag)


Original Y_report shape: (8816, 59)
Unique counts per label:
 report_sinus_rhythm                         2
report_sinus_bradycardia                    2
report_sinus_tachycardia                    2
report_atrial_fibrillation                  2
report_atrial_flutter                       2
report_supraventricular_tachycardia         2
report_ventricular_tachycardia              2
report_idioventricular_rhythm               1
report_bundle_branch_block                  1
report_right_bundle_branch_block            2
report_left_bundle_branch_block             2
report_av_block                             1
report_left_anterior_fascicular_block       2
report_left_posterior_fascicular_block      2
report_intraventricular_conduction_delay    1
report_aberrant_conduction                  2
report_left_ventricular_hypertrophy         2
report_right_ventricular_hypertrophy        2
report_biatrial_enlargement                 2
report_left_atrial_enlargement              2
report_right_atria

In [6]:
Xr_train

,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,ED Heart Rate,ED sbp,ED dbp,ED Respitory Rate,ED Oxygen Saturation %,ED Temperature (F),minutes_since_vitals,anchor_age,gender,race
1,674,290,29999,499,598,899,58,262,92,98.0,170.0,116.0,20.0,94.0,98.2,12.0,60.0,M,WHITE
2,667,286,29999,499,597,886,43,268,103,98.0,170.0,116.0,20.0,94.0,98.2,50.0,60.0,M,WHITE
3,698,292,29999,499,601,893,11,-62,103,89.0,147.0,90.0,16.0,96.0,NaN,27.0,60.0,M,WHITE
19,822,351,29999,505,609,872,30,-18,2,84.0,167.0,92.0,16.0,100.0,98.7,65.0,65.0,F,WHITE
36,606,300,29999,502,617,860,53,-32,-11,101.0,116.0,72.0,16.0,98.0,98.4,8.0,66.0,M,ASIAN - CHINESE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75173,789,350,29999,506,600,902,50,12,44,95.0,109.0,68.0,18.0,100.0,NaN,235.0,49.0,F,BLACK/AFRICAN AMERICAN
75176,577,437,29999,499,567,970,58,53,-59,102.0,96.0,45.0,19.0,97.0,98.6,6.0,NaN,F,BLACK/AFRICAN AMERICAN
75177,769,353,29999,505,599,904,72,24,59,86.0,137.0,88.0,18.0,99.0,98.1,11.0,49.0,F,BLACK/AFRICAN AMERICAN
75188,545,359,29999,511,567,862,53,10,32767,110.0,119.0,74.0,18.0,98.0,97.8,7.0,49.0,F,BLACK/AFRICAN AMERICAN


In [7]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
# -----------------------------
# 7. Preprocessing
# -----------------------------
from sklearn.impute import SimpleImputer

def make_preprocessor(numeric_features):
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), demo_categorical),
            ('num', SimpleImputer(strategy='median'), numeric_features)  # Fill NaNs with median
        ]
    )


prep_report = make_preprocessor(ecg_numeric + vital_features + demo_numeric)
prep_diag = make_preprocessor(ecg_numeric + vital_features + demo_numeric)

Xr_train_enc = prep_report.fit_transform(Xr_train)
Xr_test_enc = prep_report.transform(Xr_test)

Xd_train_enc = prep_diag.fit_transform(Xd_train)
Xd_test_enc = prep_diag.transform(Xd_test)

# -----------------------------
# 8. Train MultiOutput XGBoost
# -----------------------------
def train_multilabel_xgb(X, Y):
    base = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    model = MultiOutputClassifier(base, n_jobs=-1)
    model.fit(X, Y)
    return model

report_model = train_multilabel_xgb(Xr_train_enc, Yr_train)
diagnosis_model = train_multilabel_xgb(Xd_train_enc, Yd_train)

/home/syamala/.local/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/syamala/.local/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [8]:
data

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,bandwidth,filtering,rr_interval,...,label_ventricular_arrhythmias,is_cardiovascular,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,minutes_since_vitals
1,10101340,42195598,42195598,2110-01-18 09:41:00,files/p1010/p10101340/s42195598/42195598,22311569.0,35530181.0,0.0005-150 Hz,<not specified>,674,...,0,1,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0,12.0
2,10101340,48109767,48109767,2110-01-18 10:19:00,files/p1010/p10101340/s48109767/48109767,22311569.0,35530181.0,0.0005-150 Hz,<not specified>,667,...,0,1,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0,50.0
3,10101340,48220829,48220829,2110-01-18 14:47:00,files/p1010/p10101340/s48220829/48220829,22311569.0,35530181.0,0.0005-150 Hz,<not specified>,698,...,0,1,2110-01-18 14:20:00,NaN,89.0,16.0,96.0,147.0,90.0,27.0
19,11576763,46786216,46786216,2110-02-16 01:37:00,files/p1157/p11576763/s46786216/46786216,25798110.0,31377710.0,0.0005-150 Hz,<not specified>,822,...,0,0,2110-02-16 00:32:00,98.7,84.0,16.0,100.0,167.0,92.0,65.0
36,14609638,48429717,48429717,2110-03-13 10:29:00,files/p1460/p14609638/s48429717/48429717,26669049.0,34890898.0,0.0005-150 Hz,<not specified>,606,...,0,1,2110-03-13 10:21:00,98.4,101.0,16.0,98.0,116.0,72.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75173,12648465,46900074,46900074,2209-05-02 00:11:00,files/p1264/p12648465/s46900074/46900074,22009525.0,32057880.0,0.0005-150 Hz,<not specified>,789,...,0,1,2209-05-01 20:16:00,NaN,95.0,18.0,100.0,109.0,68.0,235.0
75176,13233424,42873676,42873676,2209-05-23 15:34:00,files/p1323/p13233424/s42873676/42873676,NaN,34904273.0,0.0005-150 Hz,<not specified>,577,...,0,0,2209-05-23 15:28:00,98.6,102.0,19.0,97.0,96.0,45.0,6.0
75177,12648465,41799270,41799270,2209-05-26 11:04:00,files/p1264/p12648465/s41799270/41799270,20892088.0,36343811.0,0.0005-150 Hz,<not specified>,769,...,0,1,2209-05-26 10:53:00,98.1,86.0,18.0,99.0,137.0,88.0,11.0
75188,12648465,47900027,47900027,2209-08-07 12:00:00,files/p1264/p12648465/s47900027/47900027,22702738.0,37922513.0,0.0005-150 Hz,<not specified>,545,...,0,1,2209-08-07 11:53:00,97.8,110.0,18.0,98.0,119.0,74.0,7.0


In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_multilabel(model, X_test_enc, Y_test, model_name="Model"):
    # 1. Make predictions
    Y_pred = model.predict(X_test_enc)
    
    # 2. Overall subset accuracy (exact match)
    subset_acc = accuracy_score(Y_test, Y_pred)
    
    # 3. F1 scores
    f1_micro = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
    f1_macro = f1_score(Y_test, Y_pred, average='macro', zero_division=0)
    
    print(f"\n===== {model_name} Overall Performance =====")
    print(f"Subset Accuracy (exact match across all labels): {subset_acc:.4f}")
    print(f"F1 Score (Micro): {f1_micro:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    
    # 4. Per-label metrics
    print(f"\n===== {model_name} Per-Label Classification Report =====")
    print(classification_report(
        Y_test,
        Y_pred,
        target_names=Y_test.columns,
        zero_division=0  # avoids warnings for labels with no positive samples
    ))
    
    # 5. Optional: Accuracy per label individually
    label_acc = {}
    for i, col in enumerate(Y_test.columns):
        label_acc[col] = accuracy_score(Y_test.iloc[:, i], Y_pred[:, i])
    print(f"\n===== {model_name} Per-Label Accuracy =====")
    for k, v in label_acc.items():
        print(f"{k}: {v:.4f}")
    
    return Y_pred, subset_acc, f1_micro, f1_macro, label_acc

# Example usage:
Yr_pred, report_subset_acc, report_f1_micro, report_f1_macro, report_label_acc = evaluate_multilabel(
    report_model, Xr_test_enc, Yr_test, model_name="Report Model"
)

Yd_pred, diag_subset_acc, diag_f1_micro, diag_f1_macro, diag_label_acc = evaluate_multilabel(
    diagnosis_model, Xd_test_enc, Yd_test, model_name="Diagnosis Model"
)


===== Report Model Overall Performance =====
Subset Accuracy (exact match across all labels): 0.6236
F1 Score (Micro): 0.7860
F1 Score (Macro): 0.2917

===== Report Model Per-Label Classification Report =====
                                        precision    recall  f1-score   support

                   report_sinus_rhythm       0.92      0.96      0.94      1047
              report_sinus_bradycardia       0.70      0.84      0.76       128
              report_sinus_tachycardia       0.87      0.94      0.90       171
            report_atrial_fibrillation       0.85      0.69      0.76       143
                 report_atrial_flutter       0.00      0.00      0.00         6
   report_supraventricular_tachycardia       0.00      0.00      0.00         4
        report_ventricular_tachycardia       0.00      0.00      0.00         0
      report_right_bundle_branch_block       0.68      0.54      0.60        63
       report_left_bundle_branch_block       0.76      0.42      0.54

In [10]:
#Trying something
# -----------------------------
# 1. Group REPORT labels (ECG)
# -----------------------------
report_label_groups = {
    # Normal / general
    'report_normal_ecg': ['report_normal_ecg'],
    'report_borderline_ecg': ['report_borderline_ecg'],
    'report_abnormal_ecg': ['report_abnormal_ecg'],
    
    # Bundle blocks / fascicular
    'bundle_branch_blocks': [
        'report_right_bundle_branch_block', 
        'report_left_bundle_branch_block', 
        'report_left_anterior_fascicular_block', 
        'report_left_posterior_fascicular_block',
        'report_aberrant_conduction'
    ],
    
    # Atrial enlargements
    'atrial_enlargement': [
        'report_left_atrial_enlargement', 
        'report_right_atrial_enlargement', 
        'report_biatrial_enlargement'
    ],
    
    # Arrhythmias
    'sinus_rhythms': [
        'report_sinus_rhythm',
        'report_sinus_bradycardia',
        'report_sinus_tachycardia',
        'report_sinus_arrhythmia'
    ],
    'atrial_fib_flutter': ['report_atrial_fibrillation', 'report_atrial_flutter'],
    'supraventricular_tachycardia': ['report_supraventricular_tachycardia'],
    'ventricular_tachycardia': ['report_ventricular_tachycardia', 'report_ventricular_bigeminy','report_ventricular_trigeminy'],
    
    # Misc
    'axis_deviation': ['report_axis_deviation'],
    'prolonged_qt': ['report_prolonged_qt'],
    'technical_error': ['report_technical_error'],
    'lead_reversal': ['report_lead_reversal'],
    'wpw_pattern': ['report_wpw_pattern']
}

def group_labels(Y, label_groups):
    new_Y = pd.DataFrame(index=Y.index)
    for group_name, cols in label_groups.items():
        present_cols = [c for c in cols if c in Y.columns]
        if present_cols:
            # if any of the grouped labels is 1 → group = 1
            new_Y[group_name] = Y[present_cols].max(axis=1)
    return new_Y

Yr_train_grouped = group_labels(Yr_train, report_label_groups)
Yr_test_grouped = group_labels(Yr_test, report_label_groups)

# -----------------------------
# 2. Group DIAGNOSIS labels (diagnoses)
# -----------------------------
diag_label_groups = {
    'acute_heart_failure': ['label_acute_heart_failure', 'label_acute_on_chronic_heart_failure'],
    'chronic_heart_failure': ['label_chronic_heart_failure'],
    'atrial_arrhythmias': ['label_atrial_fibrillation_flutter', 'label_other_arrhythmias'],
    'ventricular_arrhythmias': ['label_ventricular_arrhythmias', 'label_supraventricular_tachycardia'],
    'kidney_issues': ['label_acute_kidney_injury','label_chronic_kidney_disease','label_dialysis_dependence'],
    'hypertension': ['label_hypertension', 'label_hypertensive_emergency'],
    'ischemic_events': ['label_stable_angina','label_unstable_angina','label_stemi','label_subsequent_mi'],
    'other': [col for col in Yd_train.columns if col not in [
        'label_acute_heart_failure','label_acute_on_chronic_heart_failure','label_chronic_heart_failure',
        'label_atrial_fibrillation_flutter','label_other_arrhythmias','label_ventricular_arrhythmias',
        'label_supraventricular_tachycardia','label_acute_kidney_injury','label_chronic_kidney_disease','label_dialysis_dependence',
        'label_hypertension','label_hypertensive_emergency','label_stable_angina','label_unstable_angina','label_stemi','label_subsequent_mi'
    ]]
}

Yd_train_grouped = group_labels(Yd_train, diag_label_groups)
Yd_test_grouped = group_labels(Yd_test, diag_label_groups)

In [11]:
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

def train_multilabel_chain(X, Y):
    base = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        use_label_encoder=False
    )
    
    chain = ClassifierChain(base, order='random', random_state=42)
    chain.fit(X, Y)
    return chain

# Preprocessing stays the same
prep_report = make_preprocessor(ecg_numeric + vital_features + demo_numeric)
prep_diag = make_preprocessor(ecg_numeric + vital_features + demo_numeric)

Xr_train_enc = prep_report.fit_transform(Xr_train)
Xr_test_enc = prep_report.transform(Xr_test)

Xd_train_enc = prep_diag.fit_transform(Xd_train)
Xd_test_enc = prep_diag.transform(Xd_test)

/home/syamala/.local/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/syamala/.local/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [12]:
# Train with grouped labels
report_model_chain = train_multilabel_chain(Xr_train_enc, Yr_train_grouped)
diagnosis_model_chain = train_multilabel_chain(Xd_train_enc, Yd_train_grouped)

/home/syamala/.local/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [21:01:05] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/syamala/.local/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [21:01:06] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/syamala/.local/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [21:01:07] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/syamala/.local/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [21:01:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/syamala/.local/lib/python3.11/site-packages/xgboost/tr

In [13]:
import numpy as np
import pandas as pd

Yr_pred = report_model_chain.predict(Xr_test_enc)
Yd_pred = diagnosis_model_chain.predict(Xd_test_enc)

# Convert to DataFrames for readability
Yr_pred = pd.DataFrame(Yr_pred, columns=Yr_test_grouped.columns, index=Yr_test_grouped.index)
Yd_pred = pd.DataFrame(Yd_pred, columns=Yd_test_grouped.columns, index=Yd_test_grouped.index)



In [14]:
from sklearn.metrics import accuracy_score

def per_label_accuracy(Y_true, Y_pred):
    return pd.Series(
        {col: accuracy_score(Y_true[col], Y_pred[col]) for col in Y_true.columns}
    ).sort_values()

report_label_acc = per_label_accuracy(Yr_test_grouped, Yr_pred)
diag_label_acc   = per_label_accuracy(Yd_test_grouped, Yd_pred)

print("Report label accuracy:")
print(report_label_acc)

print("\nDiagnosis label accuracy:")
print(diag_label_acc)


Report label accuracy:
bundle_branch_blocks            0.922336
sinus_rhythms                   0.933107
report_borderline_ecg           0.937642
report_abnormal_ecg             0.960884
atrial_fib_flutter              0.963152
atrial_enlargement              0.969955
prolonged_qt                    0.969955
axis_deviation                  0.971655
report_normal_ecg               0.981293
ventricular_tachycardia         0.996032
supraventricular_tachycardia    0.997732
technical_error                 1.000000
dtype: float64

Diagnosis label accuracy:
kidney_issues              0.770975
hypertension               0.772676
other                      0.821995
atrial_arrhythmias         0.859410
acute_heart_failure        0.867347
chronic_heart_failure      0.931406
ventricular_arrhythmias    0.969388
ischemic_events            0.993197
dtype: float64


In [15]:
from sklearn.metrics import accuracy_score

report_exact_match = accuracy_score(Yr_test_grouped, Yr_pred)
diag_exact_match   = accuracy_score(Yd_test_grouped, Yd_pred)

print("Report exact-match accuracy:", report_exact_match)
print("Diagnosis exact-match accuracy:", diag_exact_match)


Report exact-match accuracy: 0.717687074829932
Diagnosis exact-match accuracy: 0.38945578231292516


In [16]:
from sklearn.metrics import f1_score, hamming_loss

def multilabel_metrics(Y_true, Y_pred, name="Model"):
    print(f"\n{name} performance:")
    print("Hamming loss:", hamming_loss(Y_true, Y_pred))
    print("Micro F1:", f1_score(Y_true, Y_pred, average='micro'))
    print("Macro F1:", f1_score(Y_true, Y_pred, average='macro'))
    print("Weighted F1:", f1_score(Y_true, Y_pred, average='weighted'))

multilabel_metrics(Yr_test_grouped, Yr_pred, "Report chain")
multilabel_metrics(Yd_test_grouped, Yd_pred, "Diagnosis chain")



Report chain performance:
Hamming loss: 0.03302154195011338
Micro F1: 0.8436591366584657
Macro F1: 0.5409949895298087
Weighted F1: 0.825450693551291

Diagnosis chain performance:
Hamming loss: 0.12670068027210885
Micro F1: 0.7038754554488241
Macro F1: 0.3734100224296463
Weighted F1: 0.6690597146676035


In [23]:
Yr_pred_chain = pd.DataFrame(
    report_model_chain.predict(Xr_test_enc),
    columns=Yr_test_grouped.columns,
    index=Yr_test_grouped.index
)

Yd_pred_chain = pd.DataFrame(
    diagnosis_model_chain.predict(Xd_test_enc),
    columns=Yd_test_grouped.columns,
    index=Yd_test_grouped.index
)



In [24]:
print(Yr_pred_chain.shape)
print(Yr_test_grouped.shape)


(1764, 12)
(1764, 12)


In [25]:
from sklearn.metrics import classification_report, f1_score

print("===== Report Model (ClassifierChain) =====")
print(classification_report(
    Yr_test_grouped,
    Yr_pred_chain,
    zero_division=0
))

print("Micro F1:",
      f1_score(Yr_test_grouped, Yr_pred_chain, average="micro"))
print("Macro F1:",
      f1_score(Yr_test_grouped, Yr_pred_chain, average="macro"))


===== Report Model (ClassifierChain) =====
              precision    recall  f1-score   support

           0       0.57      0.46      0.51        37
           1       0.46      0.58      0.51        99
           2       0.86      0.78      0.82       201
           3       0.69      0.48      0.57       186
           4       0.00      0.00      0.00        53
           5       0.93      0.99      0.96      1359
           6       0.83      0.70      0.76       149
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         7
           9       0.74      0.69      0.71        90
          10       0.85      0.53      0.65        94
          11       1.00      1.00      1.00         2

   micro avg       0.86      0.83      0.84      2281
   macro avg       0.58      0.52      0.54      2281
weighted avg       0.83      0.83      0.83      2281
 samples avg       0.85      0.82      0.82      2281

Micro F1: 0.8436591366584657
Macro F

In [26]:
from sklearn.metrics import classification_report, f1_score

print("===== Diagnosis Model (ClassifierChain) =====")
print(classification_report(
    Yd_test_grouped,
    Yd_pred_chain,
    zero_division=0
))

print("Micro F1:",
      f1_score(Yd_test_grouped, Yd_pred_chain, average="micro"))
print("Macro F1:",
      f1_score(Yd_test_grouped, Yd_pred_chain, average="macro"))

===== Diagnosis Model (ClassifierChain) =====
              precision    recall  f1-score   support

           0       0.46      0.27      0.34       223
           1       0.00      0.00      0.00       118
           2       0.72      0.42      0.53       334
           3       0.00      0.00      0.00        52
           4       0.56      0.39      0.46       439
           5       0.77      0.81      0.79       921
           6       0.00      0.00      0.00        12
           7       0.85      0.89      0.87      1136

   micro avg       0.76      0.66      0.70      3235
   macro avg       0.42      0.35      0.37      3235
weighted avg       0.70      0.66      0.67      3235
 samples avg       0.52      0.48      0.47      3235

Micro F1: 0.7038754554488241
Macro F1: 0.3734100224296463


In [27]:
len(REPORT_LABELS)

59

In [28]:
len(DIAGNOSIS_LABELS)

66

In [ ]:
#The classifier chain substantially improves recall and F1 score by modeling label dependencies, enabling detection of co-occurring diagnoses at the cost of some precision.